# 02 · Distribution Fitting & Monte Carlo Simulation

Fit **Normal** and **Student-t** distributions to S&P 500 daily log returns, compare via Q-Q plots, then run a simple Monte Carlo price simulation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
import config

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(config.RANDOM_STATE)
plt.rcParams.update({'figure.dpi': config.FIG_DPI})

## 1. Load Returns

In [ ]:
df = pd.read_csv(
    os.path.join(config.DATA_DIR, 'sp500_processed.csv'),
    index_col='Date', parse_dates=True
)
returns = df['LogReturn'].dropna()
print(f'Return series length: {len(returns):,}')
print(returns.describe().round(6))

## 2. Fit Normal Distribution

In [ ]:
def fit_normal(data: pd.Series) -> dict:
    """Fit a Normal distribution to data and return parameters."""
    mu, sigma = stats.norm.fit(data)
    return {'dist': stats.norm, 'params': (mu, sigma),
            'mu': mu, 'sigma': sigma}

norm_fit = fit_normal(returns)
print(f"Normal  μ={norm_fit['mu']:.6f}  σ={norm_fit['sigma']:.6f}")

## 3. Fit Student-t Distribution

In [ ]:
def fit_student_t(data: pd.Series) -> dict:
    """Fit a Student-t distribution (df, loc, scale) to data."""
    df_t, loc, scale = stats.t.fit(data)
    return {'dist': stats.t, 'params': (df_t, loc, scale),
            'df': df_t, 'loc': loc, 'scale': scale}

t_fit = fit_student_t(returns)
print(f"Student-t  df={t_fit['df']:.2f}  loc={t_fit['loc']:.6f}  scale={t_fit['scale']:.6f}")

## 4. Histogram with Fitted Densities

In [ ]:
fig, ax = plt.subplots(figsize=config.FIG_SIZE)
x = np.linspace(returns.min(), returns.max(), 500)

ax.hist(returns, bins=80, density=True, color='lightsteelblue',
        edgecolor='white', alpha=0.7, label='Empirical')

# Normal
pdf_norm = stats.norm.pdf(x, *norm_fit['params'])
ax.plot(x, pdf_norm, 'tomato', linewidth=2, label='Normal fit')

# Student-t
pdf_t = stats.t.pdf(x, *t_fit['params'])
ax.plot(x, pdf_t, 'seagreen', linewidth=2, linestyle='--', label='Student-t fit')

ax.set_title('Daily Log Return Distribution – Normal vs Student-t')
ax.set_xlabel('Log Return')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Q-Q Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normal Q-Q
(osm_n, osr_n), (slope_n, intercept_n, r_n) = stats.probplot(returns, dist='norm')
axes[0].scatter(osm_n, osr_n, s=6, alpha=0.5, color='steelblue')
line_n = slope_n * np.array([osm_n[0], osm_n[-1]]) + intercept_n
axes[0].plot([osm_n[0], osm_n[-1]], line_n, 'tomato', linewidth=2)
axes[0].set_title(f'Q-Q Plot vs Normal  (R²={r_n**2:.3f})')
axes[0].set_xlabel('Theoretical Quantiles')
axes[0].set_ylabel('Sample Quantiles')

# Student-t Q-Q
(osm_t, osr_t), (slope_t, intercept_t, r_t) = stats.probplot(
    returns, dist='t', sparams=(t_fit['df'],))
axes[1].scatter(osm_t, osr_t, s=6, alpha=0.5, color='steelblue')
line_t = slope_t * np.array([osm_t[0], osm_t[-1]]) + intercept_t
axes[1].plot([osm_t[0], osm_t[-1]], line_t, 'seagreen', linewidth=2)
axes[1].set_title(f'Q-Q Plot vs Student-t  (R²={r_t**2:.3f})')
axes[1].set_xlabel('Theoretical Quantiles')
axes[1].set_ylabel('Sample Quantiles')

plt.suptitle('Q-Q Plots: Empirical Returns vs Fitted Distributions', y=1.01)
plt.tight_layout()
plt.show()

print(f'Normal R²={r_n**2:.4f}   Student-t R²={r_t**2:.4f}')

## 6. Goodness-of-Fit – Kolmogorov-Smirnov Test

In [ ]:
def ks_test(data, fit_info):
    stat, pval = stats.kstest(data, fit_info['dist'].cdf, args=fit_info['params'])
    return stat, pval

ks_n_stat, ks_n_p = ks_test(returns, norm_fit)
ks_t_stat, ks_t_p = ks_test(returns, t_fit)
print(f'KS Normal:    statistic={ks_n_stat:.4f}  p-value={ks_n_p:.4e}')
print(f'KS Student-t: statistic={ks_t_stat:.4f}  p-value={ks_t_p:.4e}')

## 7. Monte Carlo Price Simulation

Simulate 252 trading days (≈1 year) forward from the last observed close using parameters drawn from the fitted Student-t distribution.

In [ ]:
def monte_carlo_simulation(
    last_price: float,
    n_days: int,
    n_sims: int,
    dist_fit: dict,
    seed: int = config.RANDOM_STATE,
) -> np.ndarray:
    """
    Return array of shape (n_days, n_sims) with simulated price paths.
    Uses fitted distribution parameters for daily log-return sampling.
    """
    rng = np.random.default_rng(seed)
    d = dist_fit
    if d['dist'] == stats.t:
        simulated_returns = rng.standard_t(df=d['df'], size=(n_days, n_sims)) * d['scale'] + d['loc']
    else:  # Normal
        simulated_returns = rng.normal(loc=d['mu'], scale=d['sigma'], size=(n_days, n_sims))

    # Cumulative price paths
    log_cum = np.cumsum(simulated_returns, axis=0)
    paths = last_price * np.exp(log_cum)
    return paths


last_close = df['Close'].iloc[-1]
N_SIMS = 500
N_DAYS = 252

paths = monte_carlo_simulation(last_close, N_DAYS, N_SIMS, t_fit)

fig, ax = plt.subplots(figsize=config.FIG_SIZE)
ax.plot(paths[:, :100], linewidth=0.5, alpha=0.4, color='steelblue')
p5  = np.percentile(paths, 5,  axis=1)
p50 = np.percentile(paths, 50, axis=1)
p95 = np.percentile(paths, 95, axis=1)
ax.plot(p50, color='black',  linewidth=2, label='Median')
ax.fill_between(range(N_DAYS), p5, p95, color='steelblue', alpha=0.15, label='5–95th pct')
ax.set_title(f'Monte Carlo Simulation – {N_SIMS:,} Paths, {N_DAYS} Days (Student-t)')
ax.set_xlabel('Trading Days Ahead')
ax.set_ylabel('Simulated S&P 500 Price')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Starting price : {last_close:,.2f}')
print(f'Median end price: {p50[-1]:,.2f}')
print(f'5th  pct end  : {p5[-1]:,.2f}')
print(f'95th pct end  : {p95[-1]:,.2f}')